In [4]:
# ============================================
# K-Means Clustering with NLP on data_02.csv
# ============================================

# Install required libraries (run once if needed)
# !pip install pandas numpy matplotlib seaborn scikit-learn nltk wordcloud

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# NLP libraries
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re

# Machine learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Download NLP resources
nltk.download('punkt')
nltk.download('stopwords')


# ============================================
# 1. Load Dataset
# ============================================

df = pd.read_csv("data_02.csv")

# Display dataset information
print(df.head())
print(df.info())


# ============================================
# 2. Select Text Column
# ============================================

# Change 'text' to your actual text column name
text_column = 'text'

documents = df[text_column].astype(str)


# ============================================
# 3. NLP Text Cleaning
# ============================================

stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Lowercase
    text = text.lower()
    
    # Remove special characters and numbers
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Tokenization
    words = word_tokenize(text)
    
    # Remove stopwords
    words = [
        word for word in words 
        if word not in stop_words
    ]
    
    return " ".join(words)


df['clean_text'] = documents.apply(clean_text)

df[['text','clean_text']].head()


# ============================================
# 4. Convert Text to TF-IDF Features
# ============================================

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X = tfidf.fit_transform(df['clean_text'])

print("TF-IDF Shape:", X.shape)


# ============================================
# 5. Find Optimal Number of Clusters
# ============================================

silhouette_scores = []

K_range = range(2,11)

for k in K_range:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    labels = model.fit_predict(X)
    
    score = silhouette_score(X, labels)
    silhouette_scores.append(score)


plt.figure(figsize=(8,5))
plt.plot(K_range, silhouette_scores, marker='o')
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Optimal K Selection")
plt.show()


# ============================================
# 6. Apply K-Means
# ============================================

optimal_k = 5   # Change based on graph

kmeans = KMeans(
    n_clusters=optimal_k,
    random_state=42,
    n_init=10
)

df['cluster'] = kmeans.fit_predict(X)


# View cluster distribution
print(df['cluster'].value_counts())


# ============================================
# 7. Display Important Words per Cluster
# ============================================

terms = tfidf.get_feature_names_out()

for i in range(optimal_k):
    
    cluster_words = kmeans.cluster_centers_[i]
    
    top_words = cluster_words.argsort()[-10:][::-1]
    
    print("\nCluster", i)
    print([terms[word] for word in top_words])


# ============================================
# 8. Visualise Clusters using PCA
# ============================================

pca = PCA(n_components=2)

X_pca = pca.fit_transform(X.toarray())


plt.figure(figsize=(8,6))

sns.scatterplot(
    x=X_pca[:,0],
    y=X_pca[:,1],
    hue=df['cluster'],
    palette="viridis"
)

plt.title("K-Means NLP Clusters")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")

plt.show()


# ============================================
# 9. Save Results
# ============================================

df.to_csv(
    "data_02_clustered.csv",
    index=False
)

print("Completed. Clustered file saved.")

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


ParserError: Error tokenizing data. C error: Expected 2 fields in line 326, saw 3
